# 📐 Modelo TCO Sostenible (Escenario MITMA 2026 - 44 Toneladas)

Este análisis desglosa la estructura de costes de explotación para un vehículo articulado de gran tonelaje, cumpliendo con la metodología del **Observatorio de Costes del Ministerio de Transportes**.

La tarifa técnica se calcula mediante la suma de costes fijos (amortizados por km) y costes variables directos:

$$ \text{Tarifa Técnica} (€/km) = \text{Coste Fijo Km} + \text{Coste Variable Km} $$

### 1. Coste Fijo por Kilómetro
Calculado sobre la base anual de gastos que no dependen del movimiento del vehículo:
$$ \text{Coste Fijo Km} = \frac{\sum (\text{Personal} + \text{Amortización} + \text{Seguros} + \text{Indirectos})}{\text{Kilometraje Anual}} $$

### 2. Coste Variable por Kilómetro
Costes directos de la operación por cada unidad de distancia:
$$ \text{Coste Variable Km} = \text{Combustible} + \text{Neumáticos} + \text{Mantenimiento} + \text{AdBlue} $$

In [47]:
import numpy as np
from scipy.optimize import minimize

# 1. Configuración de Tarifas y Objetivo
TARIFA_INTERNA_VENTA = 1.50
TARIFA_MERCADO_REF = 1.70
TARGET_TCO_DIESEL = TARIFA_INTERNA_VENTA * 0.85  # Buscamos que el coste sea el 85% del precio

# 2. Parámetros Operativos (Línea Base Diésel)
KM_CONSIDERADOS = 125_000   # Optimizado para máxima productividad
FIJOS_ESTRUCTURALES = 4_000 + 12_000   # Seguros + Indirectos
VAR_OPERATIVOS = 0.440 + 0.140 + 0.006 # Combustible + Mantenimiento + AdBlue

def objetivo_estrategia_diesel(x):
    coste_fijo_total = FIJOS_ESTRUCTURALES + x[0] + x[1]
    tarifa_resultante = (coste_fijo_total / KM_CONSIDERADOS) + VAR_OPERATIVOS
    return (tarifa_resultante - TARGET_TCO_DIESEL)**2

# 3. Límites de búsqueda (Bounds)
bounds_diesel = [(44_000, 55_000), (16_000, 20_000)]

# 4. Cálculo
initial_guess = [50_000, 18_500]
res_opt = minimize(objetivo_estrategia_diesel, initial_guess, bounds=bounds_diesel, method='SLSQP')

if res_opt.success:
    o_pers, o_fin = res_opt.x
    tco_real = ((FIJOS_ESTRUCTURALES + o_pers + o_fin) / KM_CONSIDERADOS) + VAR_OPERATIVOS
    
    print(f"=== ANÁLISIS FINANCIERO DE LA DIVISIÓN (DIÉSEL) ===")
    print("-" * 55)
    print(f"TARIFA FINAL DE VENTA:      {TARIFA_INTERNA_VENTA:.2f} €/km")
    print(f"Coste de Explotación (TCO):  {tco_real:.4f} €/km")
    print("-" * 55)
    print(f"MARGEN NETO GENERADO:       {(TARIFA_INTERNA_VENTA - tco_real):.4f} €/km")
    print(f"RENTABILIDAD SOBRE VENTA:   {((1 - tco_real/TARIFA_INTERNA_VENTA)*100):.2f} %")
    print("-" * 55)
    print(f"Personal Anual:              {o_pers:,.0f} €")
    print(f"Alquiler/Financiación Anual: {o_fin:,.0f} €")
    print(f"Productividad Necesaria:     {KM_CONSIDERADOS:,.0f} km/año")
    print("-" * 55)
    print(f"AHORRO VS MERCADO (1.70€):   {(TARIFA_MERCADO_REF - TARIFA_INTERNA_VENTA):.2f} €/km")
else:
    print("No se encontró solución. Intenta subir los KM anuales.")

=== ANÁLISIS FINANCIERO DE LA DIVISIÓN (DIÉSEL) ===
-------------------------------------------------------
TARIFA FINAL DE VENTA:      1.50 €/km
Coste de Explotación (TCO):  1.2620 €/km
-------------------------------------------------------
MARGEN NETO GENERADO:       0.2380 €/km
RENTABILIDAD SOBRE VENTA:   15.87 %
-------------------------------------------------------
Personal Anual:              50,000 €
Alquiler/Financiación Anual: 18,500 €
Productividad Necesaria:     125,000 km/año
-------------------------------------------------------
AHORRO VS MERCADO (1.70€):   0.20 €/km


## 🔋 Escenario: Camión Eléctrico (44 Toneladas)

A continuación, modelamos el **Total Cost of Ownership (TCO)** para un vehículo eléctrico de gran tonelaje, considerando los mayores costes de inversión (CAPEX) pero menores costes operativos (OPEX).

In [49]:
import numpy as np
from scipy.optimize import minimize

# 1. Configuración de Tarifas (Escenario Eléctrico Premium)
TARIFA_INTERNA_EV = 1.60      # Tarifa de venta superior por ser tecnología Green
TARGET_TCO_EV = TARIFA_INTERNA_EV * 0.85  # Objetivo: 1.36 €/km (15% margen)

# 2. Parámetros Bloqueados (Línea Base Eléctrica)
KM_CONSIDERADOS = 110_000     # Utilización estándar
FIJOS_ESTRUCTURALES_EV = 3_500 + 5_500 # Infraestructura + Seguros/Visados
# OPEX Eléctrico: 1.35 kWh/km @ 0.12 €/kWh + 0.09 mantenimiento
VAR_OPERATIVOS_EV = (1.35 * 0.12) + 0.090 

def objetivo_estrategia_ev(x):
    """
    x[0] -> personal_y_dietas
    x[1] -> financiacion_vehiculo_ev
    """
    coste_fijo_total = FIJOS_ESTRUCTURALES_EV + x[0] + x[1]
    tarifa_resultante = (coste_fijo_total / KM_CONSIDERADOS) + VAR_OPERATIVOS_EV
    return (tarifa_resultante - TARGET_TCO_EV)**2

# 3. Límites de búsqueda (Bounds)
# Salario: 44k-55k | Renting EV: 30k-45k (ajustado a CAPEX real)
bounds_ev = [(44_000, 55_000), (30_000, 45_000)]

# 4. Cálculo
initial_guess = [50_000, 38_000]
res_opt_ev = minimize(objetivo_estrategia_ev, initial_guess, bounds=bounds_ev, method='SLSQP')

if res_opt_ev.success:
    o_pers, o_fin = res_opt_ev.x
    tco_real_ev = ((FIJOS_ESTRUCTURALES_EV + o_pers + o_fin) / KM_CONSIDERADOS) + VAR_OPERATIVOS_EV
    
    print(f"=== ANÁLISIS FINANCIERO: DIVISIÓN ELÉCTRICA (GREEN) ===")
    print("-" * 60)
    print(f"TARIFA DE VENTA (Sostenible): {TARIFA_INTERNA_EV:.2f} €/km")
    print(f"Coste de Explotación (TCO):   {tco_real_ev:.4f} €/km")
    print("-" * 60)
    print(f"MARGEN NETO GENERADO:         {(TARIFA_INTERNA_EV - tco_real_ev):.4f} €/km")
    print(f"RENTABILIDAD SOBRE VENTA:     {((1 - tco_real_ev/TARIFA_INTERNA_EV)*100):.2f} %")
    print("-" * 60)
    print(f"Personal y Dietas:             {o_pers:,.0f} €/año")
    print(f"Cuota Renting EV:              {o_fin:,.0f} €/año (aprox. {o_fin/12:,.0f} €/mes)")
    print("-" * 60)
    print(f"Diferencial vs Mercado (1.70€): {(1.70 - TARIFA_INTERNA_EV):.2f} €/km de ahorro")
else:
    print("No se encontró solución. Los costes fijos del eléctrico son muy altos para esa cifra.")


=== ANÁLISIS FINANCIERO: DIVISIÓN ELÉCTRICA (GREEN) ===
------------------------------------------------------------
TARIFA DE VENTA (Sostenible): 1.60 €/km
Coste de Explotación (TCO):   1.1338 €/km
------------------------------------------------------------
MARGEN NETO GENERADO:         0.4662 €/km
RENTABILIDAD SOBRE VENTA:     29.14 %
------------------------------------------------------------
Personal y Dietas:             50,000 €/año
Cuota Renting EV:              38,000 €/año (aprox. 3,167 €/mes)
------------------------------------------------------------
Diferencial vs Mercado (1.70€): 0.10 €/km de ahorro
